In [1]:
import pandas as pd
import numpy as np
import os
import statsmodels.api as sm
from tqdm.notebook import tqdm

c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# Directories
PROJECT = "C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/"
DATA = os.path.join(PROJECT, "Data")

# Input files
# ALL_TWEETS_AGGREGATED = "C:/Users/skazempour/Documents/StockTwits/dataset/v1/data/aggregated/aggregated_sentiment_returns.pkl"
INPUT_DATA = os.path.join(DATA, "merged_master.pkl")
LINEAR_REGRESSION_PREDICTIONS = os.path.join(DATA, "predictions_linear_regression.pkl")
# DECISION_TREE_PREDICTIONS = os.path.join(DATA, "predictions_decision_tree.pkl")
# DECISION_TREE_SHALLOW_PREDICTIONS = os.path.join(DATA, "predictions_decision_tree_shallow.pkl")
# DECISION_TREE_MODERATE_PREDICTIONS = os.path.join(DATA, "predictions_decision_tree_moderate.pkl")
# DECISION_TREE_CONSTRAINED_PREDICTIONS = os.path.join(DATA, "predictions_decision_tree_constrained.pkl")
# DECISION_TREE_CV_PRUNED_PREDICTIONS = os.path.join(DATA, "predictions_decision_tree_cv_pruned.pkl")
# RANDOM_FOREST_PREDICTIONS = os.path.join(DATA, "predictions_random_forest.pkl")
# NEURAL_NETWORK_PREDICTIONS = os.path.join(DATA, "predictions_neural_network.pkl")
LINEAR_REGRESSION_ALL_FEATURES_PREDICTIONS = os.path.join(DATA, "predictions_linear_regression_all_features.pkl")

# Put together all predictions

In [3]:
# Load the original aggregated tweets data
master_data = pd.read_pickle(INPUT_DATA)
master_data = master_data[master_data['date'] >= '2012-01-01']

# Add the linear regression model
linear_regression_predictions = pd.read_pickle(LINEAR_REGRESSION_PREDICTIONS).drop(columns=['index', 'ticker'])
linear_regression_predictions.columns = ['date', 'permno', 'lr_exp', 'lr_roll_252', 'lr_roll_21']
df = pd.merge(master_data, linear_regression_predictions, on=['date', 'permno']) # Clean merge

# # Add the decision tree model without prunning
# decision_tree_predictions = pd.read_pickle(DECISION_TREE_PREDICTIONS).drop(columns=['index'])
# decision_tree_predictions.columns = ['date', 'symbol', 'dt_exp', 'dt_roll_252', 'dt_roll_21']
# df = pd.merge(df, decision_tree_predictions, on=['date', 'symbol']) # Clean merge

# # Add the shallow decision tree model
# decision_tree_shallow_predictions = pd.read_pickle(DECISION_TREE_SHALLOW_PREDICTIONS).drop(columns=['index'])
# decision_tree_shallow_predictions.columns = ['date', 'symbol', 'dt_shallow_exp', 'dt_shallow_roll_252', 'dt_shallow_roll_21']
# df = pd.merge(df, decision_tree_shallow_predictions, on=['date', 'symbol']) # Clean merge

# # Add the moderate decision tree model
# decision_tree_moderate_predictions = pd.read_pickle(DECISION_TREE_MODERATE_PREDICTIONS).drop(columns=['index'])
# decision_tree_moderate_predictions.columns = ['date', 'symbol', 'dt_moderate_exp', 'dt_moderate_roll_252', 'dt_moderate_roll_21']
# df = pd.merge(df, decision_tree_moderate_predictions, on=['date', 'symbol']) # Clean merge

# # Add the constrained decision tree model
# decision_tree_constrained_predictions = pd.read_pickle(DECISION_TREE_CONSTRAINED_PREDICTIONS).drop(columns=['index'])
# decision_tree_constrained_predictions.columns = ['date', 'symbol', 'dt_constrained_exp', 'dt_constrained_roll_252', 'dt_constrained_roll_21']
# df = pd.merge(df, decision_tree_constrained_predictions, on=['date', 'symbol']) # Clean merge

# # Add the cv pruned decision tree model
# decision_tree_cv_pruned_predictions = pd.read_pickle(DECISION_TREE_CV_PRUNED_PREDICTIONS).drop(columns=['index'])
# decision_tree_cv_pruned_predictions.columns = ['date', 'symbol', 'dt_cv_pruned_exp', 'dt_cv_pruned_roll_252', 'dt_cv_pruned_roll_21']
# df = pd.merge(df, decision_tree_cv_pruned_predictions, on=['date', 'symbol']) # Clean merge

# # Add the random forest model
# random_forest_predictions = pd.read_pickle(RANDOM_FOREST_PREDICTIONS).drop(columns=['index'])
# random_forest_predictions.columns = ['date', 'symbol', 'rf_exp', 'rf_roll_252', 'rf_roll_21']
# df = pd.merge(df, random_forest_predictions, on=['date', 'symbol']) # Clean merge

# # Add the neural network model
# neural_network_predictions = pd.read_pickle(NEURAL_NETWORK_PREDICTIONS).drop(columns=['index'])
# neural_network_predictions.columns = ['date', 'symbol', 'nn_exp', 'nn_roll_252', 'nn_roll_21']
# df = pd.merge(df, neural_network_predictions, on=['date', 'symbol']) # Clean merge

# Add the linear regression model with all features
linear_regression_all_features_predictions = pd.read_pickle(LINEAR_REGRESSION_ALL_FEATURES_PREDICTIONS).drop(columns=['ticker'])
linear_regression_all_features_predictions.columns = ['date', 'permno', 'lr_all_features_exp', 'lr_all_features_roll_252', 'lr_all_features_roll_21']
df = pd.merge(df, linear_regression_all_features_predictions, on=['date', 'permno']) # Clean merge

# Run sentiment regressions

In [4]:
prediction_columns = ['lr_exp', 'lr_roll_252', 'lr_roll_21',
                      'lr_all_features_exp', 'lr_all_features_roll_252', 'lr_all_features_roll_21']

ret_col = 'ar_FF5_1'
reg_results = []
for pred_col in tqdm(prediction_columns):
    reg_data = df[['permno','date', ret_col, pred_col]].dropna().copy()
    reg_data['date'] = pd.to_datetime(reg_data['date'])
    reg_data['date'] = reg_data['date'].dt.year * 10000 + reg_data['date'].dt.month*100 + reg_data['date'].dt.day 
    reg_data = reg_data.rename(columns={pred_col: 'pred'})
    y = reg_data[ret_col]
    X = sm.add_constant(reg_data['pred'])
    model = sm.OLS(y, X, missing='drop').fit(cov_type='cluster',cov_kwds={'groups':np.array(reg_data[['permno','date']])})
    reg_results.append(model)

  0%|          | 0/6 [00:00<?, ?it/s]

# Render the results in a table

In [5]:
from latex_table import linear_regression

# Format and save the table
vars_to_include = ['pred', 'const']
var_names = ["Prediction", "Const."]
rename_dict = dict(zip(vars_to_include, var_names))

tbl = linear_regression(reg_results)
tbl.rename_variables(rename_dict)
tbl.columns = prediction_columns
tbl.obs = True
tbl.R2 = True
tbl.float_format = ".2f"
tbl.render(midrule=True)
tbl.tbl


lr_exp    lr_roll_252     lr_roll_21 lr_all_features_exp  \
Prediction          0.70  0.78THREESTAR  0.42THREESTAR       0.50THREESTAR   
                  (0.44)         (0.11)         (0.07)              (0.15)   
Const.             -0.00           0.00           0.00                0.00   
                  (0.00)         (0.00)         (0.00)              (0.00)   
                                                                             
N             14,557,219     14,557,219     14,557,219          14,557,219   

             lr_all_features_roll_252 lr_all_features_roll_21  
Prediction                0.38TWOSTAR          -0.00THREESTAR  
                               (0.15)                  (0.00)  
Const.                           0.00             0.00ONESTAR  
                               (0.00)                  (0.00)  
                                               ADDMIDRULEHERE  
N                          14,557,219              14,557,219

In [10]:
for r in reg_results:
    print(r.rsquared)

2.2882771031351723e-06
8.270766015072706e-05
0.0001078720116851617
5.2105237233535107e-05
7.182842430797365e-05
1.8479881234156892e-06


In [9]:
reg_results[0].rsquared

2.2882771031351723e-06